# CURE-Rec — next actions and paper-readiness notebook

This notebook is the execution checklist after the master run. It separates **cheap postprocessing and validation actions** from **expensive model-search and multi-seed actions**.

## Required order

1. verify the completed Torch BPR run and audit;
2. run the staged validation-only BPR search;
3. inspect the selected final BPR/hybrid configuration;
4. regenerate aggregate assets from completed expensive CURE sweeps without rerunning them;
5. regenerate the controlled oracle regime suite with separate estimated/oracle recovery metrics;
6. archive the reproducibility snapshot;
7. only then plan calibration robustness and SASRec.


## 1. Setup and source verification

Run this cell after `git pull` and a kernel restart. It deliberately clears stale `cure_rec` modules from long-lived VS Code kernels.

In [1]:
from pathlib import Path
import hashlib
import importlib
import inspect
import json
import shutil
import sys
import pandas as pd

CWD = Path.cwd().resolve()
CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the CURE-Rec code directory or repository root.')
sys.path[:] = [str(ROOT), *[item for item in sys.path if item != str(ROOT)]]
for name in list(sys.modules):
    if name == 'cure_rec' or name.startswith('cure_rec.'):
        del sys.modules[name]
importlib.invalidate_caches()

from cure_rec.analysis import analyze_dataset
from cure_rec.config import load_settings
from cure_rec.data import load_dataset
from cure_rec.experiments import postprocess_seed_sweep
from cure_rec.regimes import run_regime_suite
from cure_rec.observability import RunLogger
from cure_rec.search import SearchConfig, run_final_bpr_audit, run_final_bpr_seed_replication, run_staged_bpr_search
from cure_rec.models import chronological_leave_one_out

assert 'bpr_epochs' in inspect.signature(analyze_dataset).parameters
print('CURE-Rec source:', ROOT)
print('Analysis signature:', inspect.signature(analyze_dataset))


CURE-Rec source: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code
Analysis signature: (result: 'DatasetLoadResult', *, output_root: 'str | Path', run_bpr: 'bool' = True, bpr_updates: 'int' = 500000, bpr_epochs: 'int' = 200, bpr_backend: 'str' = 'auto', max_eval_users: 'int' = 1000, seed: 'int' = 42) -> 'DataAnalysisResult'


In [2]:
import inspect

from cure_rec.search import run_final_bpr_seed_replication

source = inspect.getsource(run_final_bpr_seed_replication)

assert "paired = bpr.merge" in source
assert "pop.loc[row.seed" not in source

print("Final BPR seed replication code is current")

Final BPR seed replication code is current


## 2. Configure paths to the completed results

The default paths point to your currently committed runs. Change them only when you intentionally want to inspect a different run.

In [3]:
RUN_ROOT = ROOT / 'runs'
MOVIELENS_SOURCE = ROOT / 'data' / 'raw' / 'movielens_1m'
TORCH_RUN = RUN_ROOT / 'data-analysis-movielens_1m-20260805T102506Z'
MASTER_RUN = RUN_ROOT / 'all-variations-20260804T202725Z'
FULL_FIVE_SWEEP = MASTER_RUN / 'full_five_seed' / 'seed-sweep-20260804T210624Z'
FULL_TWENTY_SWEEP = MASTER_RUN / 'full_twenty_seed' / 'seed-sweep-20260805T000401Z'
QUICK_CONFIG = ROOT / 'configs' / 'curesim_quickstart.yaml'
FULL_CONFIG = ROOT / 'configs' / 'curesim_full.yaml'

for path in [TORCH_RUN, MASTER_RUN, FULL_FIVE_SWEEP, FULL_TWENTY_SWEEP]:
    print(('FOUND' if path.exists() else 'MISSING'), path)


FOUND /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/runs/data-analysis-movielens_1m-20260805T102506Z
FOUND /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/runs/all-variations-20260804T202725Z
FOUND /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/runs/all-variations-20260804T202725Z/full_five_seed/seed-sweep-20260804T210624Z
FOUND /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/runs/all-variations-20260804T202725Z/full_twenty_seed/seed-sweep-20260805T000401Z


## 3. Action 1 — inspect the completed Torch BPR run

This is cheap and should be done before any new search. The critical audit violations must all be zero. Candidate coverage must be shared across popularity, Torch BPR, and hybrid.

In [14]:
if not TORCH_RUN.exists():
    raise FileNotFoundError(f'Completed Torch run not found: {TORCH_RUN}')

metrics = pd.read_csv(TORCH_RUN / 'tables' / 'data_table_model_metrics.csv')
eval_audit = pd.read_csv(TORCH_RUN / 'tables' / 'data_table_evaluation_audit.csv')
pairwise = pd.read_csv(TORCH_RUN / 'tables' / 'data_table_pairwise_accuracy.csv')
hybrid_validation = pd.read_csv(TORCH_RUN / 'tables' / 'data_table_hybrid_validation.csv')
manifest = json.loads((TORCH_RUN / 'artifacts' / 'analysis_manifest.json').read_text())

display(metrics)
display(eval_audit)
display(pairwise)
display(hybrid_validation.sort_values('ndcg_at_k', ascending=False))
print('Manifest backend:', manifest.get('bpr_backend'))
print('Selected hybrid alpha:', manifest.get('selected_hybrid_alpha'))


,model,evaluated_users,candidate_coverage,cold_test_items,recall_at_k,ndcg_at_k,hit_rate_at_k
0,popularity,1000,0.999001,1,0.049,0.025520,0.049
1,torch_bpr_mf_bias,1000,0.999001,1,0.052,0.024127,0.052
2,bpr_popularity_hybrid_alpha_0.40,1000,0.999001,1,0.049,0.025737,0.049


,model,evaluated_users,warm_test_targets,cold_test_targets,mean_candidate_count,minimum_candidate_count,maximum_candidate_count,seen_item_violations,missing_target_violations,candidate_equality_violations,descending_score_violations,best_validation_epoch,restored_checkpoint_epoch
0,popularity,1000,1000,1,3436.958,2739,3521,0,0,0,0,NaN,NaN
1,torch_bpr_mf_bias,1000,1000,1,3436.958,2739,3521,0,0,0,0,2.0,2.0
2,hybrid_alpha_0.40,1000,1000,1,3436.958,2739,3521,0,0,0,0,NaN,NaN


,model,pairwise_training_accuracy,training_pairs,pairwise_validation_accuracy,validation_pairs
0,popularity,0.8681,10000,0.824241,6031
1,torch_bpr_mf_bias,0.8759,10000,0.826894,6031
2,hybrid_alpha_0.40,0.8685,10000,0.824905,6031


,alpha,model,evaluated_users,candidate_coverage,cold_test_items,recall_at_k,ndcg_at_k,hit_rate_at_k
1,0.40,hybrid_alpha_0.40,1000,0.999001,1,0.060,0.030982,0.060
2,0.55,hybrid_alpha_0.55,1000,0.999001,1,0.059,0.030862,0.059
0,0.25,hybrid_alpha_0.25,1000,0.999001,1,0.060,0.030687,0.060
3,0.70,hybrid_alpha_0.70,1000,0.999001,1,0.059,0.030672,0.059
4,0.85,hybrid_alpha_0.85,1000,0.999001,1,0.060,0.030354,0.060
5,1.00,hybrid_alpha_1.00,1000,0.999001,1,0.058,0.029040,0.058


Manifest backend: torch
Selected hybrid alpha: 0.4


## 4. Action 2 — enforce the evaluation audit gate

Do not proceed to search or SASRec if this cell fails. The assertions establish that metric differences are not produced by candidate leakage, missing targets, or ranking-direction errors.

In [15]:
critical = [
    'seen_item_violations',
    'missing_target_violations',
    'candidate_equality_violations',
    'descending_score_violations',
]
for column in critical:
    assert (eval_audit[column] == 0).all(), f'Audit failure in {column}'

for _, row in eval_audit.dropna(subset=['best_validation_epoch']).iterrows():
    assert row['restored_checkpoint_epoch'] == row['best_validation_epoch'], 'Best checkpoint was not restored'

print('Evaluation audit passed.')
print('Candidate coverage:', metrics[['model', 'candidate_coverage', 'cold_test_items']].to_dict(orient='records'))
print('Pairwise diagnostics:', pairwise.to_dict(orient='records'))


Evaluation audit passed.
Candidate coverage: [{'model': 'popularity', 'candidate_coverage': 0.999000999000999, 'cold_test_items': 1}, {'model': 'torch_bpr_mf_bias', 'candidate_coverage': 0.999000999000999, 'cold_test_items': 1}, {'model': 'bpr_popularity_hybrid_alpha_0.40', 'candidate_coverage': 0.999000999000999, 'cold_test_items': 1}]
Pairwise diagnostics: [{'model': 'popularity', 'pairwise_training_accuracy': 0.8681, 'training_pairs': 10000, 'pairwise_validation_accuracy': 0.8242414193334439, 'validation_pairs': 6031}, {'model': 'torch_bpr_mf_bias', 'pairwise_training_accuracy': 0.8759, 'training_pairs': 10000, 'pairwise_validation_accuracy': 0.8268943790416183, 'validation_pairs': 6031}, {'model': 'hybrid_alpha_0.40', 'pairwise_training_accuracy': 0.8685, 'training_pairs': 10000, 'pairwise_validation_accuracy': 0.8249046592604875, 'validation_pairs': 6031}]


## 5. Action 3 — staged Torch BPR search

Enable this only after the audit gate passes. Stage A searches optimization and negative strategy; Stage B searches capacity and batch size; Stage C selects hybrid alpha using validation only. Test metrics are evaluated once after configuration selection.

In [6]:
RUN_STAGED_BPR_SEARCH = True
SEARCH_OUTPUT = RUN_ROOT / 'bpr-search-movielens-final'

if RUN_STAGED_BPR_SEARCH:
    ml1m = load_dataset('movielens_1m', MOVIELENS_SOURCE, download=True)
    split = chronological_leave_one_out(ml1m.interactions)
    final_search = run_staged_bpr_search(
        split,
        SEARCH_OUTPUT,
        SearchConfig(stage_epochs=40, final_epochs=200, max_eval_users=1_000, top_k_stage_a=3, seed=42),
    )
    print('Search output:', SEARCH_OUTPUT)
    display(final_search)
else:
    print('Staged BPR search disabled. Set RUN_STAGED_BPR_SEARCH = True when ready.')


Search output: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/runs/bpr-search-movielens-final


,model,selected_config_hash,best_epoch,evaluated_users,candidate_coverage,cold_test_items,recall_at_k,ndcg_at_k,hit_rate_at_k
0,torch_bpr_mf_bias,9875430c235b,22,1000,0.999001,1,0.088,0.043566,0.088
1,hybrid_alpha_1.00,9875430c235b,22,1000,0.999001,1,0.088,0.043566,0.088


## 6. Action 4 — inspect staged search output

Run this after Stage 3 completes. The final result should be selected by validation NDCG@10, not by test metrics.

In [7]:
if SEARCH_OUTPUT.exists() and (SEARCH_OUTPUT / 'bpr_search_final_test.csv').exists():
    stage_a = pd.read_csv(SEARCH_OUTPUT / 'bpr_search_stage_a.csv')
    stage_b = pd.read_csv(SEARCH_OUTPUT / 'bpr_search_stage_b.csv')
    stage_c = pd.read_csv(SEARCH_OUTPUT / 'bpr_search_stage_c.csv')
    final_test = pd.read_csv(SEARCH_OUTPUT / 'bpr_search_final_test.csv')
    search_manifest = json.loads((SEARCH_OUTPUT / 'bpr_search_manifest.json').read_text())
    display(stage_a.head(10))
    display(stage_b.head(10))
    display(stage_c)
    display(final_test)
    print('Selected config:', search_manifest)
else:
    print('No completed staged search found yet.')


,stage,config_hash,embedding_dim,batch_size,learning_rate,weight_decay,negative_strategy,model,evaluated_users,candidate_coverage,cold_test_items,recall_at_k,ndcg_at_k,hit_rate_at_k
0,A,6d443b8fb846,64,4096,0.010,0.00001,hard_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.110,0.054445,0.110
1,A,e2dbd5178f3d,64,4096,0.003,0.00001,popularity_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.103,0.053120,0.103
2,A,a9b215c81ad4,64,4096,0.003,0.00001,hard_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.108,0.051807,0.108
3,A,684c986ba65a,64,4096,0.010,0.00001,popularity_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.099,0.051091,0.099
4,A,5f919f5c0413,64,4096,0.001,0.00001,hard_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.100,0.049892,0.100
5,A,1217cf0aadf9,64,4096,0.001,0.00001,popularity_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.092,0.047051,0.092
6,A,dc2211dd2fca,64,4096,0.003,0.00001,uniform,torch_bpr_mf_bias,1000,0.999001,1,0.084,0.045130,0.084
7,A,abb65a9cc9e4,64,4096,0.010,0.00001,uniform,torch_bpr_mf_bias,1000,0.999001,1,0.080,0.044270,0.080
8,A,eced18c017ad,64,4096,0.001,0.00001,uniform,torch_bpr_mf_bias,1000,0.999001,1,0.081,0.042596,0.081
9,A,5cb1e7cf0308,64,4096,0.003,0.00100,popularity_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.060,0.031613,0.060


,stage,parent_config_hash,config_hash,embedding_dim,batch_size,learning_rate,weight_decay,negative_strategy,model,evaluated_users,candidate_coverage,cold_test_items,recall_at_k,ndcg_at_k,hit_rate_at_k
0,B,e2dbd5178f3d,9875430c235b,128,8192,0.003,0.00001,popularity_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.110,0.054922,0.110
1,B,6d443b8fb846,4d645cfc2a79,64,8192,0.010,0.00001,hard_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.107,0.054606,0.107
2,B,a9b215c81ad4,ccdf3fea8b6e,128,4096,0.003,0.00001,hard_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.107,0.053990,0.107
3,B,6d443b8fb846,e062e93def5f,32,8192,0.010,0.00001,hard_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.107,0.053726,0.107
4,B,a9b215c81ad4,addfc1afc217,128,8192,0.003,0.00001,hard_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.109,0.053644,0.109
5,B,e2dbd5178f3d,1e008f043ccf,128,4096,0.003,0.00001,popularity_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.098,0.053224,0.098
6,B,e2dbd5178f3d,e2dbd5178f3d,64,4096,0.003,0.00001,popularity_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.103,0.053120,0.103
7,B,6d443b8fb846,6eefc750dbed,128,4096,0.010,0.00001,hard_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.096,0.052840,0.096
8,B,a9b215c81ad4,c8585ca8e925,128,2048,0.003,0.00001,hard_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.098,0.052806,0.098
9,B,e2dbd5178f3d,30f6bf2c9704,64,8192,0.003,0.00001,popularity_mixture,torch_bpr_mf_bias,1000,0.999001,1,0.104,0.052794,0.104


,stage,alpha,model,evaluated_users,candidate_coverage,cold_test_items,recall_at_k,ndcg_at_k,hit_rate_at_k
0,C,1.00,hybrid_alpha_1.00,1000,0.999001,1,0.110,0.054922,0.110
1,C,0.85,hybrid_alpha_0.85,1000,0.999001,1,0.111,0.054440,0.111
2,C,0.70,hybrid_alpha_0.70,1000,0.999001,1,0.104,0.052339,0.104
3,C,0.55,hybrid_alpha_0.55,1000,0.999001,1,0.080,0.041625,0.080
4,C,0.40,hybrid_alpha_0.40,1000,0.999001,1,0.063,0.036076,0.063
5,C,0.25,hybrid_alpha_0.25,1000,0.999001,1,0.060,0.033434,0.060


,model,selected_config_hash,best_epoch,evaluated_users,candidate_coverage,cold_test_items,recall_at_k,ndcg_at_k,hit_rate_at_k
0,torch_bpr_mf_bias,9875430c235b,22,1000,0.999001,1,0.088,0.043566,0.088
1,hybrid_alpha_1.00,9875430c235b,22,1000,0.999001,1,0.088,0.043566,0.088


Selected config: {'search': {'stage_epochs': 40, 'final_epochs': 200, 'max_eval_users': 1000, 'top_k_stage_a': 3, 'seed': 42}, 'best_config': {'embedding_dim': 128, 'batch_size': 8192, 'learning_rate': 0.003, 'weight_decay': 1e-05, 'negative_strategy': 'popularity_mixture'}, 'alpha': 1.0}


## 7. Action 4A — final selected BPR audit

Run this after staged search completes. It retrains the frozen selected configuration once, restores its best validation checkpoint, and emits a dedicated final-model audit. This is the audit that belongs in the reproducibility archive.


In [8]:
RUN_FINAL_BPR_AUDIT = True
FINAL_BPR_AUDIT_OUTPUT = RUN_ROOT / 'final-bpr-audit-movielens'

if RUN_FINAL_BPR_AUDIT:
    final_ml1m = load_dataset('movielens_1m', MOVIELENS_SOURCE, download=True)
    final_split = chronological_leave_one_out(final_ml1m.interactions)
    final_bpr_audit = run_final_bpr_audit(
        final_split,
        SEARCH_OUTPUT,
        FINAL_BPR_AUDIT_OUTPUT,
        seed=42,
        max_eval_users=1_000,
    )
    print('Final BPR audit:', FINAL_BPR_AUDIT_OUTPUT)
    display(final_bpr_audit)
else:
    print('Final BPR audit disabled. Enable after accepting the staged search configuration.')


Final BPR audit: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/runs/final-bpr-audit-movielens


,seed,model,selected_config_hash,evaluated_users,candidate_coverage,cold_test_items,recall_at_k,ndcg_at_k,hit_rate_at_k,best_validation_epoch,restored_checkpoint_epoch
0,42,popularity,9875430c235b,1000,0.999001,1,0.049,0.025520,0.049,NaN,NaN
1,42,torch_bpr_mf_bias,9875430c235b,1000,0.999001,1,0.088,0.043566,0.088,22.0,22.0


## 8. Action 4B — fixed-configuration BPR seed replication

Run only after the final selected-model audit passes. This uses the frozen selected configuration without any retuning and reports seed-level paired differences against popularity.


In [1]:
# Stand-alone safety: this cell is also safe to run directly in a newly restarted kernel.
# It still uses the exact same frozen search configuration and versioned output directory.
from pathlib import Path
import importlib
import inspect
import sys

if 'ROOT' not in globals():
    CWD = Path.cwd().resolve()
    CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
    ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
    if ROOT is None:
        raise RuntimeError('Open this notebook from the CURE-Rec code directory or repository root.')

# Reload the installed source so this action cannot accidentally use a stale kernel module.
sys.path[:] = [str(ROOT), *[item for item in sys.path if item != str(ROOT)]]
for name in list(sys.modules):
    if name == 'cure_rec' or name.startswith('cure_rec.'):
        del sys.modules[name]
importlib.invalidate_caches()

from cure_rec.data import load_dataset
from cure_rec.models import chronological_leave_one_out
from cure_rec.search import run_final_bpr_seed_replication

RUN_ROOT = ROOT / 'runs'
MOVIELENS_SOURCE = ROOT / 'data' / 'raw' / 'movielens_1m'
SEARCH_OUTPUT = RUN_ROOT / 'bpr-search-movielens-final'
RUN_FINAL_BPR_SEED_REPLICATION = True
# A versioned directory prevents stale/partial aggregate tables from prior failed runs.
FINAL_BPR_SEED_OUTPUT = RUN_ROOT / 'final-bpr-seed-replication-9875430c235b'
FINAL_BPR_SEEDS = (42, 43, 44, 45, 46)

if RUN_FINAL_BPR_SEED_REPLICATION:
    source = inspect.getsource(run_final_bpr_seed_replication)
    assert 'Incomplete final BPR replication' in source, 'Stale seed-replication code: restart kernel and rerun setup cell.'
    seed_ml1m = load_dataset('movielens_1m', MOVIELENS_SOURCE, download=True)
    seed_split = chronological_leave_one_out(seed_ml1m.interactions)
    final_bpr_seeds = run_final_bpr_seed_replication(
        seed_split,
        SEARCH_OUTPUT,
        FINAL_BPR_SEED_OUTPUT,
        seeds=FINAL_BPR_SEEDS,
        max_eval_users=1_000,
    )
    if final_bpr_seeds.empty:
        raise RuntimeError('Seed replication returned no paired rows. Use the versioned output directory after restarting the kernel.')
    print('Final BPR seed replication:', FINAL_BPR_SEED_OUTPUT)
    display(final_bpr_seeds)
else:
    print('Final BPR seed replication disabled.')


Final BPR seed replication: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/runs/final-bpr-seed-replication-9875430c235b


,seed,model,selected_config_hash,evaluated_users,candidate_coverage,cold_test_items,recall_at_k,ndcg_at_k,hit_rate_at_k,best_validation_epoch,restored_checkpoint_epoch,pop_recall_at_k,pop_ndcg_at_k,delta_recall_at_k,delta_ndcg_at_k
0,42,torch_bpr_mf_bias_final,9875430c235b,1000,0.999001,1,0.088,0.043566,0.088,22.0,22.0,0.049,0.02552,0.039,0.018046
1,43,torch_bpr_mf_bias_final,9875430c235b,1000,0.999001,1,0.090,0.044012,0.090,78.0,78.0,0.049,0.02552,0.041,0.018492
2,44,torch_bpr_mf_bias_final,9875430c235b,1000,0.999001,1,0.091,0.046442,0.091,24.0,24.0,0.049,0.02552,0.042,0.020922
3,45,torch_bpr_mf_bias_final,9875430c235b,1000,0.999001,1,0.082,0.041843,0.082,18.0,18.0,0.049,0.02552,0.033,0.016323
4,46,torch_bpr_mf_bias_final,9875430c235b,1000,0.999001,1,0.089,0.043338,0.089,56.0,56.0,0.049,0.02552,0.040,0.017818


## 7. Action 5 — regenerate aggregate assets from the completed full sweeps

This is cheap. It reuses the completed raw coalition artifacts and does not repeat the expensive 5-seed or 20-seed CURE-Sim evaluations.

In [8]:
RUN_POSTPROCESS_SWEEPS = True

if RUN_POSTPROCESS_SWEEPS:
    full_settings = load_settings(FULL_CONFIG)
    for sweep in [FULL_FIVE_SWEEP, FULL_TWENTY_SWEEP]:
        if not sweep.exists():
            print('Missing sweep:', sweep)
            continue
        rebuilt = postprocess_seed_sweep(sweep, full_settings)
        print('Postprocessed:', rebuilt.run_dir)
        display(rebuilt.decisions)
        display(rebuilt.base_feasibility)
else:
    print('Postprocessing disabled.')


Postprocessed: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/runs/all-variations-20260804T202725Z/full_five_seed/seed-sweep-20260804T210624Z


,seed,cure_run_dir,mode,status,base_feasible,selected_mask,selected_interventions,lower_improvement,upper_improvement,cost,relevance_delta_lower,provider_disparity_upper,fatigue_upper,feasible,reason,action
0,42,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,repair,repair_selected,False,1,"('repeat_cap',)",0.298514,0.313069,0.05,-0.044137,0.225872,0.0,True,Base policy violates robust constraints; selec...,repair_selected
1,43,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,improvement,improve_selected,True,1,"('repeat_cap',)",0.296202,0.310991,0.05,-0.029899,0.202785,0.0,True,Base policy is feasible; selected the exact ma...,improve_selected
2,44,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,repair,repair_selected,False,1,"('repeat_cap',)",0.292761,0.305775,0.05,-0.043151,0.219900,0.0,True,Base policy violates robust constraints; selec...,repair_selected
3,45,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,repair,repair_selected,False,1,"('repeat_cap',)",0.293929,0.313733,0.05,-0.037734,0.256836,0.0,True,Base policy violates robust constraints; selec...,repair_selected
4,46,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,repair,repair_selected,False,1,"('repeat_cap',)",0.293002,0.310941,0.05,-0.036126,0.219884,0.0,True,Base policy violates robust constraints; selec...,repair_selected


,seed,base_feasible,provider_disparity_upper,provider_margin,fatigue_upper,fatigue_margin,relevance_margin,budget_margin,provider_failure,fatigue_failure
0,42,False,0.327785,-0.047785,0.492758,0.157242,0.08,0.35,True,False
1,43,True,0.259360,0.020640,0.494760,0.155240,0.08,0.35,False,False
2,44,False,0.285023,-0.005023,0.489027,0.160973,0.08,0.35,True,False
3,45,False,0.345108,-0.065108,0.495012,0.154988,0.08,0.35,True,False
4,46,False,0.295910,-0.015910,0.492762,0.157238,0.08,0.35,True,False


Postprocessed: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/runs/all-variations-20260804T202725Z/full_twenty_seed/seed-sweep-20260805T000401Z


,seed,cure_run_dir,mode,status,base_feasible,selected_mask,selected_interventions,lower_improvement,upper_improvement,cost,relevance_delta_lower,provider_disparity_upper,fatigue_upper,feasible,reason,action
0,100,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,repair,repair_selected,False,1,"('repeat_cap',)",0.293464,0.317489,0.05,-0.036459,0.211543,0.0,True,Base policy violates robust constraints; selec...,repair_selected
1,101,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,improvement,improve_selected,True,1,"('repeat_cap',)",0.296962,0.315064,0.05,-0.039198,0.222585,0.0,True,Base policy is feasible; selected the exact ma...,improve_selected
2,102,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,repair,repair_selected,False,1,"('repeat_cap',)",0.296326,0.310043,0.05,-0.047958,0.221080,0.0,True,Base policy violates robust constraints; selec...,repair_selected
3,103,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,repair,repair_selected,False,1,"('repeat_cap',)",0.296958,0.309787,0.05,-0.035093,0.228603,0.0,True,Base policy violates robust constraints; selec...,repair_selected
4,104,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,improvement,improve_selected,True,1,"('repeat_cap',)",0.292674,0.307886,0.05,-0.039238,0.211119,0.0,True,Base policy is feasible; selected the exact ma...,improve_selected
5,105,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,repair,repair_selected,False,1,"('repeat_cap',)",0.294620,0.309152,0.05,-0.036956,0.230278,0.0,True,Base policy violates robust constraints; selec...,repair_selected
6,106,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,repair,repair_selected,False,1,"('repeat_cap',)",0.292080,0.310375,0.05,-0.048313,0.244761,0.0,True,Base policy violates robust constraints; selec...,repair_selected
7,107,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,improvement,improve_selected,True,1,"('repeat_cap',)",0.297051,0.311540,0.05,-0.045638,0.190069,0.0,True,Base policy is feasible; selected the exact ma...,improve_selected
8,108,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,improvement,improve_selected,True,1,"('repeat_cap',)",0.293904,0.314090,0.05,-0.035453,0.210093,0.0,True,Base policy is feasible; selected the exact ma...,improve_selected
9,109,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,repair,repair_selected,False,1,"('repeat_cap',)",0.291926,0.312831,0.05,-0.039497,0.201983,0.0,True,Base policy violates robust constraints; selec...,repair_selected


,seed,base_feasible,provider_disparity_upper,provider_margin,fatigue_upper,fatigue_margin,relevance_margin,budget_margin,provider_failure,fatigue_failure
0,100,False,0.282508,-0.002508,0.496061,0.153939,0.08,0.35,True,False
1,101,True,0.273056,0.006944,0.494497,0.155503,0.08,0.35,False,False
2,102,False,0.327531,-0.047531,0.494827,0.155173,0.08,0.35,True,False
3,103,False,0.320139,-0.040139,0.491878,0.158122,0.08,0.35,True,False
4,104,True,0.258295,0.021705,0.494651,0.155349,0.08,0.35,False,False
5,105,False,0.306427,-0.026427,0.494123,0.155877,0.08,0.35,True,False
6,106,False,0.280394,-0.000394,0.495684,0.154316,0.08,0.35,True,False
7,107,True,0.256713,0.023287,0.496932,0.153068,0.08,0.35,False,False
8,108,True,0.254344,0.025656,0.493370,0.156630,0.08,0.35,False,False
9,109,False,0.291242,-0.011242,0.494841,0.155159,0.08,0.35,True,False


## 8. Action 6 — regenerate controlled oracle regime metrics

This is cheap. It produces the corrected separation between estimated-game recovery and oracle-game recovery, including oracle regret in the misspecified regime.

In [9]:
RUN_REGIME_REFRESH = True

if RUN_REGIME_REFRESH:
    regime_settings = load_settings(QUICK_CONFIG)
    regime_settings.run.name = 'curesim-regime-refresh'
    regime_settings.run.output_root = RUN_ROOT
    regime_logger = RunLogger(regime_settings)
    try:
        regime_refresh = run_regime_suite(regime_settings, regime_logger)
        regime_logger.close(status='completed')
    except Exception:
        regime_logger.close(status='failed')
        raise
    print('Regime refresh:', regime_refresh.run_dir)
    display(regime_refresh.summary[[
        'regime', 'expected_estimated_selected', 'oracle_selected',
        'observed_estimated_selected', 'estimated_selection_match',
        'oracle_selection_match', 'oracle_regret',
    ]])
    display(regime_refresh.attribution_recovery.groupby('regime', as_index=False).agg(
        shapley_mae=('absolute_error', 'mean'),
        sign_accuracy=('sign_correct', 'mean'),
        point_coverage=('covered_by_estimated_point_region', 'mean'),
    ))
else:
    print('Regime refresh disabled.')


2026-08-05 15:20:48,176 | INFO | run_started | {"config_hash": "24d8eb5f21994b49", "run_id": "curesim-regime-refresh-20260805T142048Z-ac65407b"}
2026-08-05 15:20:48,180 | INFO | planner_mode_resolved | {"base_feasible": true, "base_lower_improvement": 0.0, "cost": 0.0, "fatigue_upper": 0.1, "mode": "improvement", "provider_disparity_upper": 0.22, "relevance_delta_lower": 0.0}
2026-08-05 15:20:48,180 | INFO | portfolio_rejected | {"active_interventions": ["repeat_cap", "explore_slot", "tail_slot", "diversify", "novel_slot"], "coalition_mask": 31, "cost": 0.37, "fatigue_upper": 0.05, "lower_improvement": 0.23, "mode": "improvement", "provider_disparity_upper": 0.22, "relevance_delta_lower": -0.04999999999999999}
2026-08-05 15:20:48,181 | INFO | portfolio_rejected | {"active_interventions": ["explore_slot", "tail_slot", "diversify", "provider_balance"], "coalition_mask": 46, "cost": 0.36, "fatigue_upper": 0.1, "lower_improvement": 0.2, "mode": "improvement", "provider_disparity_upper": 0.

,regime,expected_estimated_selected,oracle_selected,observed_estimated_selected,estimated_selection_match,oracle_selection_match,oracle_regret
0,additive,repeat_cap;explore_slot;tail_slot;provider_bal...,repeat_cap;explore_slot;tail_slot;provider_bal...,repeat_cap;explore_slot;tail_slot;provider_bal...,True,True,0.00
1,complementary,repeat_cap;explore_slot,repeat_cap;explore_slot,repeat_cap;explore_slot,True,True,0.00
2,redundant,tail_slot,tail_slot,tail_slot,True,True,0.00
3,antagonistic,explore_slot,explore_slot,explore_slot,True,True,0.00
4,delayed_fatigue_short,,,,True,True,0.00
5,delayed_fatigue_long,repeat_cap,repeat_cap,repeat_cap,True,True,0.00
6,provider_repair_balancing,provider_balance,provider_balance,provider_balance,True,True,0.00
7,provider_repair_repeat,repeat_cap,repeat_cap,repeat_cap,True,True,0.00
8,misspecified_ambiguity,repeat_cap,explore_slot,repeat_cap,True,False,0.19


,regime,shapley_mae,sign_accuracy,point_coverage
0,additive,0.00,1.000000,1.000000
1,antagonistic,0.00,1.000000,1.000000
2,complementary,0.00,1.000000,1.000000
3,delayed_fatigue_long,0.00,1.000000,1.000000
4,delayed_fatigue_short,0.00,1.000000,1.000000
5,misspecified_ambiguity,0.06,0.666667,0.666667
6,provider_repair_balancing,0.00,1.000000,1.000000
7,provider_repair_repeat,0.00,1.000000,1.000000
8,redundant,0.00,1.000000,1.000000


## 9. Action 7 — archive the reproducibility snapshot

This cell copies the small metadata, summary tables, and hashes into an archive folder. It intentionally does not duplicate raw MovieLens data or the entire large result tree. Use the archive as the local precursor to a Zenodo/OSF/release snapshot.

In [ ]:
# Stand-alone archive action: it can be run after Action 4B, or directly in a fresh kernel.
from pathlib import Path
import hashlib
import shutil

if 'ROOT' not in globals():
    CWD = Path.cwd().resolve()
    CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
    ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
    if ROOT is None:
        raise RuntimeError('Open this notebook from the CURE-Rec code directory or repository root.')

RUN_ROOT = ROOT / 'runs'
TORCH_RUN = RUN_ROOT / 'data-analysis-movielens_1m-20260805T102506Z'
MASTER_RUN = RUN_ROOT / 'all-variations-20260804T202725Z'
SEARCH_OUTPUT = RUN_ROOT / 'bpr-search-movielens-final'
FINAL_BPR_AUDIT_OUTPUT = RUN_ROOT / 'final-bpr-audit-movielens'
FINAL_BPR_SEED_OUTPUT = RUN_ROOT / 'final-bpr-seed-replication-9875430c235b'
ARCHIVE_RESULTS = True
ARCHIVE_DIR = ROOT.parent / 'results' / 'reproducibility_snapshot_latest'

if ARCHIVE_RESULTS:
    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
    include = [
        TORCH_RUN / 'artifacts' / 'analysis_manifest.json',
        TORCH_RUN / 'tables' / 'data_table_model_metrics.csv',
        TORCH_RUN / 'tables' / 'data_table_evaluation_audit.csv',
        TORCH_RUN / 'tables' / 'data_table_pairwise_accuracy.csv',
        MASTER_RUN / 'all_variations_summary.csv',
        MASTER_RUN / 'all_variations_seed_decisions.csv',
        SEARCH_OUTPUT / 'bpr_search_manifest.json',
        SEARCH_OUTPUT / 'bpr_search_stage_a.csv',
        SEARCH_OUTPUT / 'bpr_search_stage_b.csv',
        SEARCH_OUTPUT / 'bpr_search_stage_c.csv',
        SEARCH_OUTPUT / 'bpr_search_final_test.csv',
        # Frozen selected-model audit: configuration, scores, evaluator, checkpoint history.
        FINAL_BPR_AUDIT_OUTPUT / 'final_bpr_config.json',
        FINAL_BPR_AUDIT_OUTPUT / 'final_bpr_test_metrics.csv',
        FINAL_BPR_AUDIT_OUTPUT / 'final_bpr_evaluation_audit.csv',
        FINAL_BPR_AUDIT_OUTPUT / 'final_bpr_pairwise_accuracy.csv',
        FINAL_BPR_AUDIT_OUTPUT / 'final_bpr_loss.csv',
        FINAL_BPR_AUDIT_OUTPUT / 'final_bpr_validation.csv',
        # Fixed-configuration multi-seed replication and paired comparison to popularity.
        FINAL_BPR_SEED_OUTPUT / 'final_bpr_seed_metrics.csv',
        FINAL_BPR_SEED_OUTPUT / 'final_bpr_seed_paired_metrics.csv',
        FINAL_BPR_SEED_OUTPUT / 'final_bpr_seed_summary.csv',
    ]
    # Preserve the detailed estimated-vs-oracle recovery assets from the most
    # recent controlled-regime refresh, when it is present locally.
    regime_runs = sorted(RUN_ROOT.glob('curesim-regime-refresh-*/benchmark_regimes/*'))
    if regime_runs:
        latest_regime_run = regime_runs[-1]
        include.extend([
            latest_regime_run / 'regime_manifest.json',
            latest_regime_run / 'regime_selection_summary.csv',
            latest_regime_run / 'regime_attribution_recovery.csv',
            latest_regime_run / 'regime_interactions.csv',
            latest_regime_run / 'regime_figure_selection_recovery.png',
        ])
        print('Including controlled-regime assets:', latest_regime_run)
    else:
        print('No controlled-regime refresh found; continuing without its detailed assets.')

    checksums = []
    for source in include:
        if not source.exists():
            print('Skipping missing:', source)
            continue
        target = ARCHIVE_DIR / source.name
        shutil.copy2(source, target)
        checksums.append(f'{hashlib.sha256(target.read_bytes()).hexdigest()}  {target.name}')

    (ARCHIVE_DIR / 'SHA256SUMS.txt').write_text('\n'.join(checksums) + '\n')
    (ARCHIVE_DIR / 'REPRODUCE.md').write_text(
        'Source branch: arena/019fcbf7-next-paper\n'
        'Archive includes the external-data analysis, evaluation audit, validation-only BPR search, final frozen-model audit, fixed-configuration seed replication, controlled regimes, and CURE-Sim seed-sweep summaries.\n'
        'The archive intentionally omits raw MovieLens data and large training artifacts; use the notebook to recreate them.\n'
    )
    print('Archive written:', ARCHIVE_DIR)
else:
    print('Archive disabled. Enable after selecting final external and CURE-Sim results.')


Archive written: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/results/reproducibility_snapshot_latest


## 10. Remaining action — calibration robustness and SASRec

Do not implement or run these until the BPR search and audit are accepted. The next development task is a calibration sweep over fatigue strength, repetition threshold, horizon, provider threshold, provider-balancing strength, novelty benefit, and exploration cost. Then add SASRec using the same shared warm-item candidate evaluator and audit output.

At that point, rerun only the selected final configurations across multiple seeds; do not rerun the whole master plan blindly.